# Time-series validation vs general profiling

General-purpose data profilers summarize each column in isolation — they treat
rows as independent. **tsauditor** enforces the rules of *time*. This notebook
builds one dataset with three flaws that look clean to a general profiler but
are exactly what breaks a live model:

1. **Data-feed outage** — a clustered block of missing values (not scattered).
2. **Silent flash crash** — a value extreme *locally* but inside the global range.
3. **Data leakage** — a same-day feature that reproduces the target.

It runs with just `tsauditor` and `pandas` — no third-party profiler needed.

Check out [tsauditor](https://github.com/imann128/tsauditor) on GitHub!

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range("2026-01-01", periods=200, freq="B")
price = 100 + np.cumsum(rng.normal(0, 1, 200))

df = pd.DataFrame(
    {"Price": price, "Volume": rng.integers(10_000, 50_000, 200).astype(float)},
    index=dates,
)

# 1. Feed outage: 6 consecutive missing days in Volume (a clustered gap).
df.iloc[50:56, df.columns.get_loc("Volume")] = np.nan

# 2. Flash crash: one point 30 below its neighbours but inside the global range.
df.iloc[115, df.columns.get_loc("Price")] = price[115] - 30

# 3. Leakage: a same-day % change whose sign defines the Direction target.
ret = pd.Series(price, index=dates).pct_change()
df["ChangeP"] = ret * 100
df["Target"] = (ret > 0).astype(int)
df.head()

,Price,Volume,ChangeP,Target
2026-01-01,100.304717,47374.0,NaN,0
2026-01-02,99.264733,10557.0,-1.036825,0
2026-01-05,100.015184,16971.0,0.756010,1
2026-01-06,100.955749,19186.0,0.940422,1
2026-01-07,99.004714,21171.0,-1.932565,0


## Test 1 — a general profiler (pandas summaries)

This is what column-in-isolation profiling sees. Nothing here is wrong, exactly —
it just misses everything that matters about *time*.

In [2]:
print("Missing % per column:")
print((df.isna().mean() * 100).round(1).to_string())

print("\nPrice — global describe:")
print(df["Price"].describe().round(2).to_string())

print("\nCorrelation with Target:")
print(df.corr(numeric_only=True)["Target"].round(3).to_string())

Missing % per column:
Price      0.0
Volume     3.0
ChangeP    0.5
Target     0.0

Price — global describe:
count    200.00
mean      96.48
std        5.47
min       61.13
25%       92.10
50%       95.96
75%      101.35
max      105.48

Correlation with Target:
Price      0.117
Volume     0.030
ChangeP    0.802
Target     1.000


**What the general view concludes — and misses:**

- *Missing:* `Volume` is ~3% missing. Looks like a few scattered rows; it never
  says the feed went **dark for a solid week**.
- *Anomaly:* `Price` ranges ~70–120 globally, so a value of ~85 during the crash
  sits comfortably inside the range and raises no flag. The **local** collapse is
  invisible to a global histogram.
- *Leakage:* `ChangeP` correlates ~0.80 with `Target`, which a profiler reports as
  a "highly predictive feature" — praise for what is actually structural cheating.

## Test 2 — tsauditor (time-aware)

The same dataset, through the checks that reason about chronology.

In [3]:
from tsauditor.profiler.missing import audit_missing
from tsauditor.anomaly.contextual import audit_contextual_anomalies
from tsauditor.leakage.equivalence import audit_equivalence


def show(issues):
    for i in issues:
        print(f"  [{i.severity.upper()}] {i.code} | {i.column} | {i.description}")


print("Clustered feed outage (PRF002):")
show([i for i in audit_missing(df, domain="finance") if i.code == "PRF002"])

print("\nLocal flash crash (ANO003):")
show(
    [i for i in audit_contextual_anomalies(df, domain="finance") if i.code == "ANO003"]
)

print("\nTarget leakage (LEK001):")
show(audit_equivalence(df, target="Target"))

Clustered feed outage (PRF002):
  [WARNING] PRF002 | Volume | Column 'Volume' contains clustered missing value sequences indicating an outage.

Local flash crash (ANO003):
  [WARNING] ANO003 | Price | Contextual spikes detected.

Target leakage (LEK001):
  [CRITICAL] LEK001 | ChangeP | Feature 'ChangeP' near-deterministically reproduces target 'Target' (auc score=1.0000 >= 0.95 for binary target). Likely data leakage — review before modeling.


## Why tsauditor wins

A general profiler tells you what data *looks like* in a vacuum. tsauditor enforces
the rules of time — catching **clustered outages (PRF002)**, **contextual
disruptions (ANO003)**, and **target leakage (LEK001)** — so you don't ship a model
that passes standard validation and then fails under live conditions.

The full one-call version is just `tsa.scan(df, target="Target", domain="finance")`.